# Conquer Online Mobile — Original Client Metadata Audit
Single-cell Colab audit. Downloads the exact owner-supplied archives and updater files, verifies known hashes, and searches for Android package/version/build identifiers. Reports are saved under /content/conquer_client_metadata_audit. No game source files are changed.

In [ ]:
import requests,zipfile,io,re,json,hashlib,pathlib,csv
from datetime import datetime,timezone
ROOT=pathlib.Path("/content/conquer_client_metadata_audit"); ROOT.mkdir(exist_ok=True)
API="https://api.github.com/repos/ZzwahwahzZ/Conquer-Mobile-Assets./releases/tags/"
headers={"Accept":"application/vnd.github+json"}
releases={}
for tag in ["mobile-source-files","mobile-extracted-assets"]:
 r=requests.get(API+tag,headers=headers,timeout=90); r.raise_for_status(); releases[tag]=r.json()
 print("\nRELEASE",tag)
 for a in releases[tag]["assets"]: print(a["name"],a["size"],a["id"],a.get("digest"))
assets={a["name"]:a for rel in releases.values() for a in rel["assets"]}
known=[
("ConquerOnline.zip",461117324,"b1c6958109a07c237631943fc0442d3161563d89e1020b143d95437ba9a6fdb0","mobile-source-files"),
("Conquer-Mobile-Assets.-mobile-source-files.zip",250389993,"5acb29a128f4560bd84a956a628411f12ee713ceb5e2ecb935f65d31ba37d4e2","mobile-source-files"),
("extracted-assets-001.zip",842723106,"0d21a110c09b762ed5f1a5f4da022857b0534e489eb85052d15da460cb229d2a","mobile-extracted-assets")]
archives=[]; downloads=[]
for name,size,expected,tag in known:
 a=assets.get(name); url=a["browser_download_url"] if a else f"https://github.com/ZzwahwahzZ/Conquer-Mobile-Assets./releases/download/{tag}/{name}"
 p=ROOT/name
 try:
  if not p.exists() or p.stat().st_size!=size:
   tmp=p.with_suffix(".part")
   with requests.get(url,stream=True,timeout=(30,180)) as q:
    q.raise_for_status()
    with open(tmp,"wb") as f:
     for b in q.iter_content(8*1024*1024):
      if b:f.write(b)
   tmp.replace(p)
  h=hashlib.sha256(p.read_bytes()).hexdigest()
  if p.stat().st_size!=size or h!=expected: raise RuntimeError(f"Integrity mismatch size={p.stat().st_size} sha={h}")
  archives.append(p); downloads.append({"name":name,"status":"VERIFIED","bytes":size,"sha256":h});print("VERIFIED",name,size,h)
 except Exception as e:downloads.append({"name":name,"status":"UNAVAILABLE","error":str(e)});print("UNAVAILABLE",name,str(e)[:250])
# Download every small loose source/update asset.
loose=[]
for a in releases["mobile-source-files"]["assets"]:
 if a["size"]<=30_000_000:
  p=ROOT/a["name"]
  try:
   if not p.exists() or p.stat().st_size!=a["size"]:
    with requests.get(a["browser_download_url"],stream=True,timeout=(30,120)) as q:
     q.raise_for_status()
     with open(p,"wb") as f:
      for b in q.iter_content(1024*1024):
       if b:f.write(b)
   h=hashlib.sha256(p.read_bytes()).hexdigest()
   loose.append({"name":a["name"],"size":p.stat().st_size,"sha256":h})
   print("LOOSE VERIFIED",a["name"],p.stat().st_size,h)
  except Exception as e:print("LOOSE UNAVAILABLE",a["name"],str(e)[:180])
print("\nSCANNING ARCHIVE CONTENTS")
candidate=[]; matches=[]; apk=[]; errors=[]
exts={".apk",".aab",".obb",".zip",".ini",".dat",".cfg",".conf",".json",".xml",".txt",".properties",".manifest",".yaml",".yml",".csv",".db",".sqlite",".pak",".idx",".ver",".version"}
patterns=[("versionName",rb'versionName\s*[=:]\s*["\\\']?([A-Za-z0-9._+-]{1,80})'),("versionCode",rb'versionCode\s*[=:]\s*["\\\']?([0-9]{1,12})'),("client_version",rb'(?i)\b(?:client[_ -]?version|game[_ -]?version|app[_ -]?version)\s*[=:]\s*["\\\']?([A-Za-z0-9._+-]{1,80})'),("package",rb'(?i)\b(?:package(?:_name)?|applicationId)\s*[=:]\s*["\\\']?([A-Za-z0-9_.]{5,120})'),("build",rb'(?i)\b(?:build(?:_number|_id)?|revision|svn[_ -]?revision)\s*[=:]\s*["\\\']?([A-Za-z0-9._+-]{1,100})')]
def scan(n,b,origin,depth=0):
 if len(b)<=12_000_000:
  t=None
  for enc in ("utf-8-sig","utf-16","latin-1"):
   try:t=b.decode(enc);break
   except:pass
  if t is not None and sum(c.isprintable() or c in "\r\n\t" for c in t[:40000])/max(1,min(len(t),40000))>.82:
   found=[]
   for tag,pat in patterns:
    vals=list(dict.fromkeys(x.group(1).decode("utf8","replace") for x in re.finditer(pat,b,re.I)))[:10]
    if vals:found.append({"type":tag,"values":vals})
   if found:matches.append({"archive":origin,"path":n,"matches":found,"preview":t[:800]})
 if n.lower().endswith((".apk",".aab",".obb")):apk.append({"archive":origin,"path":n,"bytes":len(b)})
 if depth<2 and n.lower().endswith((".zip",".apk",".aab",".jar")):
  try:
   with zipfile.ZipFile(io.BytesIO(b)) as z:
    for x in z.infolist():
     if x.is_dir():continue
     if pathlib.PurePosixPath(x.filename).suffix.lower() in exts or x.filename.lower().endswith("androidmanifest.xml"):
      candidate.append({"archive":origin+"!"+n,"path":x.filename,"bytes":x.file_size})
      if x.file_size<=12_000_000:scan(x.filename,z.read(x),origin+"!"+n,depth+1)
  except Exception as e:errors.append({"file":n,"error":str(e)[:180]})
for p in archives:
 try:
  with zipfile.ZipFile(p) as z:
   print("ARCHIVE",p.name,"entries",len(z.infolist()))
   for x in z.infolist():
    if x.is_dir():continue
    if pathlib.PurePosixPath(x.filename).suffix.lower() in exts or x.filename.lower().endswith("androidmanifest.xml"):
     candidate.append({"archive":p.name,"path":x.filename,"bytes":x.file_size})
     if x.file_size<=12_000_000:scan(x.filename,z.read(x),p.name)
 except Exception as e:errors.append({"file":p.name,"error":str(e)[:200]})
for p in ROOT.iterdir():
 if p.is_file() and p.suffix.lower() not in (".zip",".part"):
  try:scan(p.name,p.read_bytes(),"loose release asset")
  except Exception as e:errors.append({"file":p.name,"error":str(e)[:150]})
report={"generated_utc":datetime.now(timezone.utc).isoformat(),"downloads":downloads,"loose_assets":loose,"metadata_candidates":candidate,"direct_metadata_matches":matches,"apk_aab_obb":apk,"errors":errors,"version_status":"UNIDENTIFIED unless explicit package/version/build evidence is found"}
(ROOT/"original-client-metadata-audit.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf8")
with open(ROOT/"original-client-metadata-candidates.csv","w",newline="",encoding="utf-8-sig") as f:
 w=csv.DictWriter(f,fieldnames=["archive","path","bytes"]);w.writeheader();w.writerows(candidate)
print("\nAUDIT RESULT")
print("Downloaded archives:",len(archives),"loose updater files:",len(loose))
print("Metadata candidates:",len(candidate),"direct metadata matches:",len(matches),"APK/AAB/OBB:",len(apk),"scan errors:",len(errors))
print("VERSION/BUILD:",report["version_status"])
print("REPORTS:",ROOT/"original-client-metadata-audit.json",ROOT/"original-client-metadata-candidates.csv")
print("This audit does not modify the game project.")